In [36]:
from pathlib import Path
import joblib
import numpy as np

#model_path = Path("D:/AHU/rdm-wach-ai/paraquet_data/models/saved/raw_data_e0202_model.pkl")
model_path = Path("D:/AHU/rdm-wach-ai/paraquet_data/models/e0202/e0202_model.pkl")
model = joblib.load(model_path)
print("Model loaded successfully!")
#D:\AHU\rdm-wach-ai\paraquet_data\models\raw_data_e0201\raw_data_e0201_test_predictions.csv

Model loaded successfully!


In [48]:
import pandas as pd

# Load Parquet
parquet_path = "D:/AHU/rdm-wach-ai/paraquet_data/cleaned/raw_data_e0202.parquet"
df = pd.read_parquet(parquet_path, engine="pyarrow")

# Optional: check first few rows
print(df.head(2))

csv_path = "D:/AHU/rdm-wach-ai/paraquet_data/cleaned/raw_data_e0202.csv"
df.to_csv(csv_path, index=False)  # index=False avoids saving the DataFrame index
print(f"CSV saved at {csv_path}")


# Last 97 rows as a DataFrame
last_97_rows = df.tail(97)

# Check shape
print(last_97_rows.shape)  # should be (97, number_of_columns)
print(last_97_rows)

                       time  result  table controller  site  power_factor_avg  \
0 2025-11-05 06:30:00+00:00     0.0    1.5      e0202   0.0              0.91   
1 2025-11-05 06:45:00+00:00     0.0    1.5      e0202   0.0              0.91   

   units  current_l1  current_l2  current_l3  volts_l1_n  volts_l2_n  \
0    0.0    4.280000    3.416667    2.770000  232.966667  231.433333   
1    0.0    4.274286    3.415714    2.781429  233.142857  231.600000   

   volts_l3_n  power_total  
0  234.100000     2.221253  
1  234.357143     2.222306  
CSV saved at D:/AHU/rdm-wach-ai/paraquet_data/cleaned/raw_data_e0202.csv
(97, 14)
                          time  result     table controller  site  \
9381 2026-02-10 23:45:00+00:00     0.0  1.500000      e0202   0.0   
9382 2026-02-11 00:00:00+00:00     0.0  1.500000      e0202   0.0   
9383 2026-02-11 00:15:00+00:00     0.0  1.538462      e0202   0.0   
9384 2026-02-11 00:30:00+00:00     0.0  1.538462      e0202   0.0   
9385 2026-02-11 00:45:00+

Load your CSV / data

In [49]:
import pandas as pd

#  Load CSV with headers
df = pd.read_csv(
    "D:/AHU/rdm-wach-ai/paraquet_data/models/raw_data_e0202/raw_data_e0202_test_predictions.csv"
)

#  Convert 'time' to datetime (optional but good for ordering)
df['time'] = pd.to_datetime(df['time'])

#  Sort by time just in case
df = df.sort_values('time')

#  Extract the 'actual' column as a list for power_total_history
power_total_history = df['actual'].tolist()

# Optional: check first 10 values
print(power_total_history[:97])

[2.13219, 2.20952, 2.211877142857143, 2.212975, 2.2139314285714287, 2.216148571428572, 2.210924285714285, 2.2221800000000003, 2.2188875, 2.225767142857143, 2.223325714285714, 2.223262857142857, 2.21153875, 2.249518571428572, 2.22743, 2.210951428571428, 2.21809625, 2.2224242857142857, 2.203794285714286, 2.2292057142857145, 2.22545375, 2.2119175, 2.22957, 2.2294114285714284, 2.2227, 2.23102, 2.2366, 2.22554, 2.23456, 2.2364475, 2.231995714285714, 2.2501366666666667, 2.25355, 2.232915, 2.2417, 2.23983, 2.2377225000000003, 2.2541725, 2.24551, 2.228377142857143, 2.244415, 2.2372428571428573, 2.22645, 2.238315714285714, 2.242405, 2.233668571428572, 2.23819, 2.2502525, 2.239395, 2.231375714285714, 2.2540475, 2.2399285714285715, 2.248415, 2.25161, 2.23747, 2.232424285714285, 2.2432275, 2.233857142857143, 2.2269050000000004, 2.239463333333333, 2.22955625, 2.22045625, 2.2322425, 2.22908, 2.22172, 2.225752857142857, 2.22598, 2.2090428571428573, 2.2373614285714285, 2.2285025, 2.227754, 2.2325325, 

Prepare current sensor history & time features

In [38]:
import pandas as pd

current_data = {
    'power_factor_avg': 0.95,  # current reading
    'current_l1': 10.5,
    'current_l2': 11.2,
    'current_l3': 9.8,
    'volts_l1_n': 230,
    'volts_l2_n': 231,
    'volts_l3_n': 229,
    'hour': 14,
    'dayofweek': 3,  # Monday=0, Sunday=6
    'month': 2,
    'is_weekend': 0  # 1 if weekend, 0 if weekday
}

Generate lag features

In [39]:
current_data['power_total_lag_1'] = power_total_history[-1]
current_data['power_total_lag_4'] = power_total_history[-4]
current_data['power_total_lag_96'] = power_total_history[-96]  # make sure you have at least 96

Prepare a buffer for lags and rolling stats

Generate rolling stattistics

In [40]:
rolling_window = 4
current_data['power_total_rolling_mean_4'] = np.mean(power_total_history[-rolling_window:])
current_data['power_total_rolling_std_4'] = np.std(power_total_history[-rolling_window:])

In [41]:
lags = [1, 4, 96]  # for power_total_lag_1, lag_4, lag_96
rolling_window = 4

Maintain a buffer of past total_power values

In [43]:
from collections import deque

# Initialize buffer with past 100 readings (or real historical data)
power_total_buffer = deque(maxlen=100)  # maxlen >= 96 for your lags

convert to dataframe

In [45]:
df = pd.DataFrame([current_data])

# Ensure the order of columns is same as training
feature_columns = [
    'power_factor_avg', 'current_l1', 'current_l2', 'current_l3',
    'volts_l1_n', 'volts_l2_n', 'volts_l3_n',
    'hour', 'dayofweek', 'month', 'is_weekend',
    'power_total_lag_1', 'power_total_lag_4', 'power_total_lag_96',
    'power_total_rolling_mean_4', 'power_total_rolling_std_4'
]

df = df[feature_columns]

Predict

In [46]:
predicted_power_total = model.predict(df)[0]
print("Predicted total_power:", predicted_power_total)

Predicted total_power: 2.2771795


Update buffer for next step

In [34]:
#if you want to predict multiple future steps (multi-step forecast)
# Append predicted value to buffer
power_total_history.append(predicted_power_total)

# Keep only last 100 (or max lag) values
power_total_history = power_total_history[-100:]

In [32]:
# feature_cols = [
#     'power_factor_avg', 'current_l1', 'current_l2', 'current_l3', 
#     'volts_l1_n', 'volts_l2_n', 'volts_l3_n', 
#     'hour', 'dayofweek', 'month', 'is_weekend', 
#     'power_total_lag_1', 'power_total_lag_4', 'power_total_lag_96', 
#     'power_total_rolling_mean_4', 'power_total_rolling_std_4'
# ]

Prepare input features for prediction

In [23]:
hour = current_timestamp.hour
dayofweek = current_timestamp.weekday()  # 0=Monday
month = current_timestamp.month
is_weekend = 1 if dayofweek >=5 else 0

NameError: name 'current_timestamp' is not defined

In [ ]:
import pandas as pd

# Example: create a single-row DataFrame for prediction
data = {
    'power_factor_avg': [0.95],
    'current_l1': [10],
    'current_l2': [9.5],
    'current_l3': [10.2],
    'volts_l1_n': [230],
    'volts_l2_n': [231],
    'volts_l3_n': [229],
    'hour': [14],
    'dayofweek': [3],
    'month': [2],
    'is_weekend': [0],
    'power_total_lag_1': [2.5],
    'power_total_lag_4': [2.6],
    'power_total_lag_96': [3.0],
    'power_total_rolling_mean_4': [2.55],
    'power_total_rolling_std_4': [0.05]
}

X = pd.DataFrame(data, columns=feature_cols)

In [13]:
y_pred = model.predict(X)
print(y_pred)

[2.319623]


for multiple rows

In [14]:
X_multi = pd.DataFrame([
    [0.95,10,9.5,10.2,230,231,229,14,3,2,0,2.5,2.6,3.0,2.55,0.05],
    [0.96,9.8,9.7,10.1,231,232,230,15,3,2,0,2.6,2.7,3.1,2.6,0.06]
], columns=feature_cols)

y_pred_multi = model.predict(X_multi)
print(y_pred_multi)

[2.319623  2.3208752]
